# 多模态与 RAG / 智能体深度融合开发

---

## 课程导航

| 部分 | 内容 | 对应文件 |
|------|------|----------|
| **Part 1** | 本节定位 · 文本优先的多模态 RAG | — |
| **Part 2** | 环境准备与模块地图 | `core/rag/*` |
| **Part 3** | 统一解析管线：多格式 → 页与块 | `core/rag/parsers.py` |
| **Part 4** | 多模态分块与元数据设计 | `core/rag/multimodal_rag.py` |
| **Part 5** | 文本优先嵌入与精排 | `core/rag/embedding.py` |
| **Part 6** | 单集合向量库 | `core/rag/vector_store.py` |
| **Part 7** | 检索编排与结果装配 | `core/rag/multimodal_rag.py` |
| **Part 8** | 与 Agent 融合：检索即工具 | `core/ai/crew_tools.py` |
| **Part 9** | 实战任务 + 随堂练习（含答案） | 多模块联合 |

---

> **面向学员**：本教程面向弱基础学员，每个知识点都有详细讲解和代码注释，
> 建议按顺序逐 Cell 运行，观察每一次的输出结果。
>
> 上一节（第 35 节）解决了"听得懂、看得懂"——把语音、截图变成文本与结构化结果。
> 本节课回答下一个问题：
> **知识从哪来、怎么被准确找到？为什么图片不直接嵌入，而要"先转成文字"？**

---
## Part 1：本节在架构中的位置 —— 文本优先的多模态 RAG
---

### 1.1 承上启下：感知之后，知识从哪来？

第 35 节的感知层把【语音 / 截图】变成了【文本 / 结构化诊断】；但数字员工要"会答题"，
还必须有一个**能查的知识库**。这就是 RAG（Retrieval-Augmented Generation，检索增强生成）。

```
感知(第35节)         本节(RAG)                      编排(第37节)
语音/截图 ──► 文本 ──► 解析→分块→嵌入→检索→装配 ──► Agent 拿"参考资料"作答
```

### 1.2 RAG 全链路快速回顾

```
┌────────┐   ┌────────┐   ┌──────────┐   ┌──────────┐   ┌────────┐   ┌──────────┐
│ ① 加载  │ → │ ② 分块  │ → │ ③ 向量化  │ → │ ④ 存储    │ → │ ⑤ 检索  │ → │ ⑥ 生成增强 │
│ PDF/Word│   │ 切片    │   │ Embedding│   │ ChromaDB │   │ 召回+精排│   │ 拼上下文送LLM│
└────────┘   └────────┘   └──────────┘   └──────────┘   └────────┘   └──────────┘
```

关键决策点只有三个：**怎么分块、用什么模型嵌入、召回多少条**。
本节课把这三点在本项目里的取舍讲透。

### 1.3 多模态 RAG 的核心难题

服务台的知识天然是**图文混排**的（报错截图、网络拓扑图、对照表格）。传统 RAG 会遇到两个尴尬：

| 难题 | 现象 | 传统做法的后果 |
|------|------|----------------|
| **文搜图命中不了** | 用户用文字问"VPN 怎么连"，命不中那张拓扑图 | 图里信息查不到 |
| **命中图却答不出** | 召回了图，但模型"看不了图"（文本 RAG 只喂文本） | 有依据却用不上 |
| **OCR 成乱序文字** | 把图 OCR 成一行行字再入库 | 版面、箭头、层级关系全丢，语义尽失 |

### 1.4 设计决策：为什么选「文本优先」

本项目的答案不是"给图片也建一套向量库"，而是**把图片先转成一段文字描述，再统一按文本嵌入**。

```
图片 ──► VLM 生成配图描述(一段中文) ──► 与正文、表格一起进【同一个文本向量空间】
图片原图 ──► 只落盘存 media/，不参与检索，命中后作为附件回传
```

| 对比项 | 图片直接嵌入 | ✅ 文本优先（本项目） |
|--------|-------------|----------------------|
| 向量空间 | 文本空间 + 图片空间（**跨模态对齐**） | **单一文本空间** |
| 集合数量 | 多集合、多维度 | **单集合、单维度** |
| 工程复杂度 | 高（要处理跨模态相似度可比性） | 低（改动收敛在检索编排） |
| "以图搜图" | 原生支持 | 降级为**语义匹配**（图→描述→文本查） |
| 换嵌入模型 | 两个空间都要重建 | 重建一个集合即可 |

> 结论：面对"IT 服务台"这种以文字答案为主、图片作辅助的场景，
> **文本优先**用一点点"以图搜图"能力，换来了整体简单、可靠、可维护。

### 1.5 本节在五层架构中的位置

| 层 | 目录 | 本节涉及的代码 |
|---|---|---|
| 路由层 | `core/api/` | `admin_api.py`：知识库上传 / 重建 / 检索测试 |
| 基础设施层 | `core/infra/` | `media.py` 落盘、`db.py` 文档元数据 |
| 智能编排层 | `core/ai/` | `crew_tools.py`：把检索封装成 Agent 工具 |
| **多模态 RAG 层** | **`core/rag/`** | **`parsers.py` / `embedding.py` / `vector_store.py` / `multimodal_rag.py`** |
| 业务编排层 | `core/service/` | `seed_knowledge.py`：演示知识库种子 |

### 1.6 两条流程：入库 与 检索

```
【入库】解析 → 分块 → 生成配图描述 → 文本嵌入 → 落库(向量+元数据)
          parsers  multimodal_rag  multimodal     embedding   vector_store

【检索】查询 → 嵌入 → 向量召回 top_k → 精排 top_n → 结果装配(文本/附件/来源)
        text/image  embedding  vector_store  reranker   multimodal_rag
```

### 1.7 三条贯穿全节的统一约定

1. **单一文本向量空间**：文本、表格 Markdown、图片配图描述全部走 `encode_text`；图片原图不嵌入。
2. **失败可降级、入库不中断**：VLM 未配置时配图描述为空串，回退页正文继续入库；精排失败回退向量召回顺序。
3. **everything 可回溯**：每个块都带 `source / page / modality`，检索结果能标注来源、图片能作为附件回传。

---
## Part 2：环境准备与模块地图
---

### 2.1 检查 Python 环境与依赖

In [1]:
# ============================================================
# Cell 1：环境与依赖检查
# ============================================================
import sys
import subprocess
import importlib

print("=" * 55)
print("  Python 环境检查")
print("=" * 55)
print(f"  Python 版本: {sys.version}")
print(f"  运行平台:   {sys.platform}")
print(f"  解释器路径: {sys.executable}")
print()

# 把项目根目录加入 import 路径，保证能 import core.*
sys.path.insert(0, '.')
print(f"  当前工作目录: {sys.path[0]}")
print()

required_packages = {
    'chromadb': 'chromadb',      # 向量数据库（嵌入式）
    'openai': 'openai',          # 统一模型客户端（嵌入走 OpenAI 兼容协议）
    'pymupdf': 'pymupdf',        # PDF 解析
    'python-docx': 'docx',       # Word 解析
    'Pillow': 'PIL',             # 图片降采样 / 编码
}

print("  依赖检查：")
for pkg_name, import_name in required_packages.items():
    try:
        importlib.import_module(import_name)
        print(f"     [OK] {pkg_name} 已安装")
    except ImportError:
        print(f"     [NO] {pkg_name} 未安装，正在安装...")
        try:
            r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg_name],
                               capture_output=True, text=True, timeout=180)
            print(f"     {'[OK] 安装成功' if r.returncode == 0 else '[NO] 安装失败: ' + r.stderr[:150]}")
        except Exception as e:
            print(f"     [NO] 安装失败: {e}")
print()
print("  环境检查完成。")

# ------------------------------------------------------------------
# 兼容补丁：让 asyncio.run() 在"已运行事件循环"的内核里也能用（幂等·可重入）
# ------------------------------------------------------------------
# 背景：部分 Jupyter / IDE 内核会把单元格代码放在一个**正在运行的事件循环**里执行，
# 此时直接调用 asyncio.run() 会报：
#     RuntimeError: asyncio.run() cannot be called from a running event loop
# 解决：包一层 run_async()——
#   · 当前线程【没有】运行中的事件循环：等价 asyncio.run（常规 Jupyter 行为不变）；
#   · 当前线程【已有】运行中的事件循环：把协程丢到"独立线程 + 新事件循环"执行，
#     再同步取回结果（线程内没有运行中的循环，故 asyncio.run 可正常启动）。
#
# 【为什么要写成幂等】本单元可能被复制/重复执行（例如编辑器把旧单元又贴回来）。
# 若把"原始实现"存成脚本全局变量，第二次执行会把它重绑成"上一次的补丁函数"，
# 于是在线程里调用它 → 自己调自己 → RecursionError。
# 故这里：① 用 asyncio 模块上的私有属性保存原始实现（不会被重绑覆盖）；
#          ② 加补丁标记 _mde_run_async_patched，重复执行直接跳过。
import asyncio
import concurrent.futures


def _install_asyncio_run_compat() -> bool:
    """安装兼容补丁；已安装则返回 False（不重复打补丁）。"""
    if getattr(asyncio, "_mde_run_async_patched", False):
        return False

    asyncio._mde_orig_run = asyncio.run   # 原始实现挂在模块属性上，后续不会被覆盖

    def run_async(coro, **kwargs):
        """在 notebook 中安全执行协程（兼容"已运行事件循环"的内核）。"""
        orig = asyncio._mde_orig_run      # 每次都取模块上的原始实现，确保不递归
        try:
            asyncio.get_running_loop()
        except RuntimeError:
            return orig(coro, **kwargs)   # 常规路径：无运行中的循环，直接跑
        # 兼容路径：已有运行中的循环 -> 换到独立线程的新循环里跑
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: orig(coro, **kwargs)).result()

    asyncio.run = run_async
    asyncio._mde_run_async_patched = True
    return True


if _install_asyncio_run_compat():
    print("已启用 run_async 兼容：asyncio.run() 现在可用于运行中的事件循环内核。")
else:
    print("run_async 兼容已启用（跳过重复补丁）。")


  Python 环境检查
  Python 版本: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
  运行平台:   win32
  解释器路径: c:\Users\13691\Documents\Code\260601-GPKJ\07-MLM\.venv\Scripts\python.exe

  当前工作目录: .

  依赖检查：
     [OK] chromadb 已安装
     [OK] openai 已安装
     [OK] pymupdf 已安装
     [OK] python-docx 已安装
     [OK] Pillow 已安装

  环境检查完成。
已启用 run_async 兼容：asyncio.run() 现在可用于运行中的事件循环内核。


> 说明：本节所有演示都**不依赖外部模型 API**——需要"嵌入"的地方，我们注入一个
> **确定性假嵌入器**（按文本哈希生成向量），这正是项目 `tests/test_rag.py` 的做法。
> 这样你即使在**断网**环境也能把全链路跑通、把设计看明白。

### 2.2 先看模块的"说明书"

每个模块的 `docstring` 就是它的"对外说明书"。先读说明书，再读代码，不容易迷路。
本节涉及 `core/rag/` 下四个模块。

In [2]:
# ============================================================
# Cell 2：四个核心模块的 docstring（为什么这样设计，写在开头）
# ============================================================
import core.rag.parsers as parsers
import core.rag.embedding as embedding
import core.rag.vector_store as vstore
import core.rag.multimodal_rag as mrag

for label, module in [
    ("core/rag/parsers.py        —— 统一解析管线", parsers),
    ("core/rag/embedding.py      —— 文本优先嵌入与精排", embedding),
    ("core/rag/vector_store.py   —— ChromaDB 单集合", vstore),
    ("core/rag/multimodal_rag.py —— 入库 / 检索 / 装配", mrag),
]:
    print("=" * 62)
    print(f"  {label}")
    print("=" * 62)
    print((module.__doc__ or "").strip())
    print()

  core/rag/parsers.py        —— 统一解析管线
文档解析：把各种格式统一解析为「页 → 文本 / 图片 / 表格」块，见设计文档 7.3 / 7.6。

支持格式：
  - ``.md`` / ``.txt``：纯文本，按标题切块（无图片、表格）；
  - ``.pdf``：PyMuPDF 逐页抽取文本 / 内嵌图片 / 表格；纯图页整页以 150dpi 渲染为图像块；
  - ``.docx``：python-docx 抽取段落文本 / 内嵌图片 / 表格（转 Markdown）；
  - ``.png`` / ``.jpg`` / ``.jpeg``：整图作为一页，一张图像块。

对外接口：:func:`parse_document`；工具函数：:func:`table_to_markdown`、:func:`downsample`。

  core/rag/embedding.py      —— 文本优先嵌入与精排
嵌入与精排 API 封装（OpenAI 兼容协议），见设计文档 7.2 / 7.4 / 第十七章。

设计要点：
  - 文本优先嵌入：文本、表格 Markdown、图片配图描述统一走 ``encode_text`` 进同一文本空间
    （Qwen3-VL-Embedding-8B）；图片本身不直接嵌入；
  - 精排走 OpenAI 兼容的 ``POST {base_url}/rerank``（Qwen3-VL-Reranker-8B）；
  - 所有同步 SDK / HTTP 调用一律用 :func:`asyncio.to_thread` 包装，避免阻塞事件循环。

对外单例：``embedder``、``reranker``。

  core/rag/vector_store.py   —— ChromaDB 单集合
ChromaDB 向量库封装（单集合、单维度），见设计文档 7.2 / 7.3 / 7.6。

采用**文本优先（完全文本化）**方案：文本 / 表格 / 图片描述都以文本嵌入映射到同一
向量空间，因此只需一个集合、一次查询即可覆盖全部模态，跨模态相似度可直接排序比较。
图片原图不参与检索，存 `data/media` 并以 ``asset_id`` 关联，检索命中后作附件回传。

对外单例：

In [3]:
# ============================================================
# Cell 3：模块地图（先有地图，再看细节）
# ============================================================
import inspect

modules = [
    ("core/rag/parsers.py", parsers),
    ("core/rag/embedding.py", embedding),
    ("core/rag/vector_store.py", vstore),
    ("core/rag/multimodal_rag.py", mrag),
]

for label, module in modules:
    members = []
    for name, obj in vars(module).items():
        if name.startswith("__"):
            continue
        if (inspect.isclass(obj) or inspect.isfunction(obj)) and getattr(obj, "__module__", None) == module.__name__:
            try:
                line = inspect.getsourcelines(obj)[1]
            except OSError:
                line = 0
            kind = "class" if inspect.isclass(obj) else "def"
            members.append((line, kind, name))
    members.sort()
    print("=" * 62)
    print(f"  {label}   （共 {len(members)} 个类 / 函数）")
    print("=" * 62)
    for line, kind, name in members:
        print(f"  [{kind:^5}] {name}   (第 {line} 行)")
    print()

print("另外三个模块级单例（实例，不是类）：")
print("  embedding.embedder / embedding.reranker   —— 嵌入与精排客户端")
print("  vstore.vector_store                       —— 进程内唯一向量库")
print("  mrag.multimodal_rag                       —— RAG 服务单例")

  core/rag/parsers.py   （共 16 个类 / 函数）
  [class] ParseError   (第 28 行)
  [class] ParsedImage   (第 35 行)
  [class] ParsedTable   (第 45 行)
  [class] ParsedPage   (第 52 行)
  [ def ] table_to_markdown   (第 66 行)
  [ def ] _encode   (第 99 行)
  [ def ] downsample   (第 106 行)
  [ def ] _split_text_sections   (第 173 行)
  [ def ] _parse_text   (第 194 行)
  [ def ] _parse_image   (第 205 行)
  [ def ] _extract_pdf_tables   (第 213 行)
  [ def ] _extract_pdf_images   (第 239 行)
  [ def ] _parse_pdf   (第 268 行)
  [ def ] _parse_docx   (第 327 行)
  [ def ] _parse_sync   (第 381 行)
  [ def ] parse_document   (第 399 行)

  core/rag/embedding.py   （共 6 个类 / 函数）
  [class] EmbeddingError   (第 31 行)
  [ def ] _is_unsupported_param_error   (第 35 行)
  [ def ] _detect_image_mime   (第 55 行)
  [ def ] build_data_url   (第 70 行)
  [class] EmbeddingClient   (第 77 行)
  [class] RerankerClient   (第 187 行)

  core/rag/vector_store.py   （共 2 个类 / 函数）
  [ def ] _sanitize_metadata   (第 23 行)
  [class] VectorStore   (第 42 行)

  

---
## Part 3：统一解析管线 —— 从"每种格式写一套"到"统一页与块"
---

### 3.1 需求来了：把公司的各种文档读进知识库

产品经理说："知识库里有 PDF、Word、Markdown，还有随手截的图，都要能吃进去。"

第 35 节的 OCR 只能抠截图里的字，**它不是文档解析**。文档解析要干的是：
把一份 PDF/Word **拆成有结构、可回溯的"页与块"**，供后续分块与嵌入。

### 3.2 v0：最直觉的写法 —— 每种格式各写一套

最容易写出来的版本，是给每种格式写一个函数，各自返回不同结构。

In [4]:
# ============================================================
# Cell 4：v0 —— 每种格式各写一套（反面教材，不真正执行）
# ============================================================
v0_code = """
def load_pdf_v0(path):
    import fitz
    doc = fitz.open(path)
    return [page.get_text() for page in doc]          # 只返回字符串，图片/表格全丢

def load_docx_v0(path):
    from docx import Document
    doc = Document(path)
    return "\\n".join(p.text for p in doc.paragraphs)   # 返回一整段，没有页码/图片

def load_img_v0(path):
    return path                                        # 只返回路径，谁去读字节？
"""
print(v0_code)
print("痛点清单：")
print("  1. 返回结构五花八门（list[str] / str / str），下游没法统一处理")
print("  2. 图片、表格被直接丢掉，多模态无从谈起")
print("  3. 没有页码 / 来源，检索命中后无法回溯")
print("  4. 新增一种格式就要改调用方")


def load_pdf_v0(path):
    import fitz
    doc = fitz.open(path)
    return [page.get_text() for page in doc]          # 只返回字符串，图片/表格全丢

def load_docx_v0(path):
    from docx import Document
    doc = Document(path)
    return "\n".join(p.text for p in doc.paragraphs)   # 返回一整段，没有页码/图片

def load_img_v0(path):
    return path                                        # 只返回路径，谁去读字节？

痛点清单：
  1. 返回结构五花八门（list[str] / str / str），下游没法统一处理
  2. 图片、表格被直接丢掉，多模态无从谈起
  3. 没有页码 / 来源，检索命中后无法回溯
  4. 新增一种格式就要改调用方


### 3.3 演进思路：定义一个统一的"页 / 块"数据模型

正确的做法是：**先定数据结构，再让每种格式的解析器都产出同一种结构**。
本项目定义了三个 `@dataclass`：

| 数据类 | 代表什么 | 关键字段 |
|--------|----------|----------|
| `ParsedPage` | 一页（或一个文本小节） | `no`（页码）、`text`、`images`、`tables`、`scanned`（是否扫描件） |
| `ParsedImage` | 一张待入库的图片块 | `id`、`data`（字节）、`ext`、`caption` |
| `ParsedTable` | 一个已结构化的表格块 | `markdown` |

> 只要所有解析器都吐出 `list[ParsedPage]`，下游就能用**同一套**分块 / 嵌入逻辑处理所有格式。

In [5]:
# ============================================================
# Cell 5：三个数据类的真实源码
# ============================================================
print(inspect.getsource(parsers.ParsedImage))
print(inspect.getsource(parsers.ParsedTable))
print(inspect.getsource(parsers.ParsedPage))
print("要点：")
print("  - ParsedPage.images / tables 用 field(default_factory=list)，避免可变默认值共享")
print("  - scanned 标记'纯图页'，供入库时判断是否走整页图嵌入")

@dataclass
class ParsedImage:
    """一张待入库的图片块。"""

    id: str
    data: bytes
    ext: str
    caption: str = ""

@dataclass
class ParsedTable:
    """一个已结构化为 Markdown 的表格块。"""

    markdown: str

@dataclass
class ParsedPage:
    """一页（或一个文本小节）的解析结果。"""

    no: int
    text: str
    images: list[ParsedImage] = field(default_factory=list)
    tables: list[ParsedTable] = field(default_factory=list)
    scanned: bool = False  # 扫描件 / 纯图页标记

要点：
  - ParsedPage.images / tables 用 field(default_factory=list)，避免可变默认值共享
  - scanned 标记'纯图页'，供入库时判断是否走整页图嵌入


### 3.4 格式 → 解析函数：一张"路由表"搞定分发

有了统一结构，还需要一个"按扩展名找解析器"的入口。本项目用一张字典 `_DISPATCH` 做路由，
再用 `_parse_sync` 统一做"文件存在性 / 类型支持"校验，最后用 `parse_document` 包一层异步。

In [6]:
# ============================================================
# Cell 6：路由表与统一入口
# ============================================================
print("  _DISPATCH：扩展名 → 解析函数")
for ext, fn in parsers._DISPATCH.items():
    print(f"    .{ext:<9} -> {fn.__name__}")
print()
print(inspect.getsource(parsers._parse_sync))
print("要点：")
print("  - 用注册表替代一长串 if/elif：新增格式只需在 _DISPATCH 加一行")
print("  - 不支持的扩展名抛 ParseError，并列出支持的格式（可操作提示）")
print("  - parse_document 用 asyncio.to_thread 包同步解析，避免阻塞事件循环")

  _DISPATCH：扩展名 → 解析函数
    .md        -> _parse_text
    .markdown  -> _parse_text
    .txt       -> _parse_text
    .pdf       -> _parse_pdf
    .docx      -> _parse_docx
    .png       -> _parse_image
    .jpg       -> _parse_image
    .jpeg      -> _parse_image
    .webp      -> _parse_image
    .bmp       -> _parse_image

def _parse_sync(file_path: str) -> list[ParsedPage]:
    path = Path(file_path)
    if not path.exists() or not path.is_file():
        raise ParseError(f"文件不存在：{file_path}")
    ext = path.suffix.lower().lstrip(".")
    parser = _DISPATCH.get(ext)
    if parser is None:
        raise ParseError(
            f"不支持的文件类型：.{ext}（支持 md/txt/pdf/docx/png/jpg/jpeg）。"
        )
    try:
        return parser(path)
    except ParseError:
        raise
    except Exception as exc:
        raise ParseError(f"解析文件失败（{path.name}）：{exc}") from exc

要点：
  - 用注册表替代一长串 if/elif：新增格式只需在 _DISPATCH 加一行
  - 不支持的扩展名抛 ParseError，并列出支持的格式（可操作提示）
  - parse_document 用 asyncio.to_thread 包同步解

### 3.5 PDF 是重头戏：一页可能有三种命运

PDF 解析是整个管线里最复杂的，因为同一份 PDF 里不同页的内容形态完全不同：

| 页面形态 | 判定 | 处理策略 | 产出 |
|----------|------|----------|------|
| **文本页** | 能抽出文本 | 抽文本 + 抽内嵌图 + 抽表格 | text + image + table 块 |
| **纯图页 / 扫描件** | 无文本层 | **整页 150dpi 渲染成一张图** | image 块（`scanned=True`） |
| **含矢量图表** | 有文本但图是矢量画的 | 依赖内嵌图抽取 / 整页兜底 | image 块 |

> 关键取舍：扫描件**不做 OCR**，而是**整页保留为图片**。
> 因为 OCR 会把"版面、箭头、层级"压成乱序文字；整页图保留了完整视觉信息，
> 由 VLM 在入库时生成配图描述来承载语义。

In [7]:
# ============================================================
# Cell 7：PDF 解析源码 _parse_pdf
# ============================================================
print(inspect.getsource(parsers._parse_pdf))
print("要点：")
print("  - 先抽文本 + 表格；两者都为空才判定为'纯图页'")
print("  - 纯图页 page.get_pixmap(dpi=150) 整页渲染，标记 scanned=True")
print("  - _extract_pdf_images 按 xref 去重，并过滤边长 < 50px 的装饰性小图")

def _parse_pdf(path: Path) -> list[ParsedPage]:
    """解析 PDF：逐页文本 + 内嵌图片 + 表格；纯图页整页渲染。"""
    try:
        import pymupdf as fitz  # PyMuPDF（1.24+ 推荐入口，避免 fitz 别名弃用告警）
    except ImportError:
        try:
            import fitz  # 兼容旧版本 PyMuPDF
        except Exception as exc:
            raise ParseError(f"PDF 解析依赖 PyMuPDF 未安装：{exc}") from exc

    try:
        doc = fitz.open(path)
    except Exception as exc:
        raise ParseError(f"PDF 打开失败：{exc}") from exc

    pages: list[ParsedPage] = []
    try:
        for page in doc:
            text = page.get_text("text") or ""
            tables = _extract_pdf_tables(page)

            if not text.strip() and not tables:
                # 纯图页 / 扫描件：整页渲染为图像块
                try:
                    pixmap = page.get_pixmap(dpi=150)
                    data = pixmap.tobytes("png")
                except Exception as exc:
                    raise ParseError(
                        f"PDF 第 {page.number + 1} 页渲染失败：{exc}"
               

In [8]:
# ============================================================
# Cell 8：真实解析两种 PDF，观察"页与块"
# ============================================================
import asyncio
from pathlib import Path

targets = [
    ("图文混合 PDF", "docs/素材/pdf/图文混合_软件安装与权限申请指南.pdf"),
    ("扫描型  PDF", "docs/素材/pdf/扫描型_VPN连接故障排查手册.pdf"),
]

for label, path in targets:
    if not Path(path).exists():
        print(f"  [跳过] 未找到 {path}")
        continue
    pages = asyncio.run(parsers.parse_document(path))
    print("=" * 62)
    print(f"  {label}：{Path(path).name}")
    print("=" * 62)
    print(f"  共解析出 {len(pages)} 页")
    for page in pages:
        print(f"    第{page.no}页 | 文本 {len(page.text):>4} 字 | "
              f"图片 {len(page.images)} 张 | 表格 {len(page.tables)} 个 | scanned={page.scanned}")
    print()

print("观察：")
print("  - 图文混合页：既有 text，又有 image / table 块")
print("  - 扫描型页：text=0、image>=1、scanned=True —— 整页被保留为一张图")

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
  图文混合 PDF：图文混合_软件安装与权限申请指南.pdf
  共解析出 2 页
    第1页 | 文本  283 字 | 图片 1 张 | 表格 0 个 | scanned=False
    第2页 | 文本  104 字 | 图片 1 张 | 表格 1 个 | scanned=False

  扫描型  PDF：扫描型_VPN连接故障排查手册.pdf
  共解析出 2 页
    第1页 | 文本    0 字 | 图片 1 张 | 表格 0 个 | scanned=True
    第2页 | 文本    0 字 | 图片 1 张 | 表格 0 个 | scanned=True

观察：
  - 图文混合页：既有 text，又有 image / table 块
  - 扫描型页：text=0、image>=1、scanned=True —— 整页被保留为一张图


### 3.6 小结：解析层的职责边界

- **只负责"拆"**：把文件拆成有结构的页与块，**不负责嵌入、不负责检索**；
- **统一产出**：所有格式 → `list[ParsedPage]`，下游不关心原始格式；
- **结构化保留**：表格转 Markdown 保住行列；扫描件整页保留保住版面。

解析完成后，下一步是把这些"块"变成可以检索的向量 —— 见 Part 4。

---
## Part 4：多模态分块与元数据设计
---

### 4.1 三类块：text / image / table

解析产出的"页"要拆成可检索的"块（chunk）"。本项目把每页的块分成三类，
每类都带一套**可回溯的元数据**：

| 块类型 `modality` | 内容载体 | 参与嵌入的文本 | 额外元数据 |
|-------------------|----------|----------------|------------|
| `text` | 正文 | 正文切片 | `content` |
| `table` | 表格 | Markdown 表格文本 | `content` |
| `image` | 图片 | **配图描述**（无描述时回退页正文 / 文件名） | `asset_id`、`caption`、`context` |

所有块都带：`source`（来源文件）、`page`（页码）、`document_id`、`category`、`chunk_index`。

> 为什么 text 要再切分？因为嵌入模型和向量检索对"长文本"不友好：
> 一段太长，向量会被稀释，检索精度下降。所以文本按固定窗口切片。

In [9]:
# ============================================================
# Cell 9：文本分块 _chunk_text（固定窗口 + 重叠）
# ============================================================
print(inspect.getsource(mrag.MultimodalRAGService._chunk_text))

demo_text = "VPN 连接失败排查。" * 200
chunks = mrag.MultimodalRAGService._chunk_text(demo_text)
print(f"  原文 {len(demo_text)} 字 → 切成 {len(chunks)} 块")
print(f"  单块上限 _CHUNK_SIZE = {mrag._CHUNK_SIZE}，重叠 _CHUNK_OVERLAP = {mrag._CHUNK_OVERLAP}")
print("要点：")
print("  - 窗口滑动步长 = chunk_size - overlap，保证相邻块'重叠'，避免答案被切断在边界")
print("  - 文本本身不超过 chunk_size 时整段作为一块（不做无谓切分）")

    @staticmethod
    def _chunk_text(
        text: str, chunk_size: int = _CHUNK_SIZE, overlap: int = _CHUNK_OVERLAP
    ) -> list[str]:
        """按固定窗口滑动切分文本（默认 500 字、重叠 50 字）。"""
        text = (text or "").strip()
        if not text:
            return []
        if len(text) <= chunk_size:
            return [text]

        step = max(1, chunk_size - overlap)
        chunks: list[str] = []
        for start in range(0, len(text), step):
            piece = text[start : start + chunk_size].strip()
            if piece:
                chunks.append(piece)
            if start + chunk_size >= len(text):
                break
        return chunks

  原文 2200 字 → 切成 5 块
  单块上限 _CHUNK_SIZE = 500，重叠 _CHUNK_OVERLAP = 50
要点：
  - 窗口滑动步长 = chunk_size - overlap，保证相邻块'重叠'，避免答案被切断在边界
  - 文本本身不超过 chunk_size 时整段作为一块（不做无谓切分）


### 4.2 图片块的"载体"选择：为什么描述优先于页正文

图片块的向量**不能是图片本身**（文本优先方案不嵌入图），必须挑一段文本来代表它。
本项目的优先级是：

```
配图描述 caption（VLM 生成，最贴合图内容）
   ↓ 没有时
页正文 context（图片所在页的文本摘要）
   ↓ 还没有时
文件名 source_name（最后的兜底，保证一定能嵌入）
```

> 这段"三级兜底"是**入库不中断**的关键：即使 VLM 没配置、图又没有任何上下文，
> 也不会因为"没有可嵌入的文本"而报错，最差用文件名兜底。

In [10]:
# ============================================================
# Cell 10：上下文提取 _find_context + 载体三级兜底
# ============================================================
print(inspect.getsource(mrag.MultimodalRAGService._find_context))
print("add_document 里图片块嵌入载体（真实代码节选）：")
print("    vector = await self.embedder.encode_text(")
print('        caption or metadata["context"] or source_name')
print("    )")
print("对照理解：caption(描述) → context(页正文) → source_name(文件名)，逐级兜底。")

    @staticmethod
    def _find_context(image: ParsedImage, page_text: str) -> str:
        """取图片所在页的文本摘要（前 200 字）作为图像块的上下文。"""
        if image.caption:
            return image.caption[:200]
        return (page_text or "")[:200]

add_document 里图片块嵌入载体（真实代码节选）：
    vector = await self.embedder.encode_text(
        caption or metadata["context"] or source_name
    )
对照理解：caption(描述) → context(页正文) → source_name(文件名)，逐级兜底。


### 4.3 实跑一次入库：用假嵌入器把多种素材写进向量库

下面复现 `tests/test_rag.py` 的做法：注入一个**确定性假嵌入器**和**关闭的精排器**，
把三种 PDF、一份 Word、一张 PNG 都入库，看三类块的数量是否符合预期。
（图片落盘用一个 stub 替换，避免污染 `data/media`。）

In [11]:
# ============================================================
# Cell 11：入库实测（假嵌入器，完全离线）
# ============================================================
import hashlib
import math
import tempfile
import pathlib
import asyncio

from core.rag.multimodal_rag import MultimodalRAGService
from core.rag.vector_store import VectorStore

DIM = 32


class FakeEmbedder:
    """确定性假嵌入器：文本按 sha256 散点成归一化向量（同样输入→同样向量）。"""

    def available(self):
        return True

    def _vec(self, seed: str):
        digest = hashlib.sha256(seed.encode("utf-8")).digest()
        raw = [b / 255.0 for b in digest]
        vec = (raw * math.ceil(DIM / len(raw)))[:DIM]
        norm = math.sqrt(sum(v * v for v in vec)) or 1.0
        return [v / norm for v in vec]

    async def encode_text(self, text):
        return self._vec(text)

    async def encode_texts(self, texts):
        return [self._vec(t) for t in texts]


class DisabledReranker:
    enabled = False

    async def rerank(self, query, candidates, top_n):
        return candidates[:top_n]


class FakeDB:
    def __init__(self):
        self.docs = {}

    async def upsert_document(self, doc_id, filename, category, modalities, chunk_count, size_bytes, status="indexed"):
        self.docs[doc_id] = {"id": doc_id, "filename": filename}

    async def delete_document(self, doc_id):
        self.docs.pop(doc_id, None)


# --- 用 stub 替换"落盘 / 文档元数据 / VLM 描述"，让演示不产生副作用、不联网 ---
_orig_db = mrag.db
_orig_save = mrag.save_media
_orig_caption = mrag.describe_knowledge_image

mrag.db = FakeDB()
_saved_assets = []

def _stub_save(data, ext, prefix="asset"):
    asset_id = f"{prefix}-nb{len(_saved_assets) + 1}"
    _saved_assets.append(asset_id)
    return asset_id

mrag.save_media = _stub_save

async def _no_caption(image_bytes):
    return ""

mrag.describe_knowledge_image = _no_caption

_tmp = tempfile.mkdtemp()
store4 = VectorStore(path=str(pathlib.Path(_tmp) / "chroma"), collection_name="nb36_part4")
service4 = MultimodalRAGService(embedder=FakeEmbedder(), reranker=DisabledReranker(), vector_store=store4)

cases = [
    ("表格型 PDF", "docs/素材/pdf/表格_IT常见错误码对照表.pdf"),
    ("图文混合 PDF", "docs/素材/pdf/图文混合_软件安装与权限申请指南.pdf"),
    ("扫描型 PDF", "docs/素材/pdf/扫描型_VPN连接故障排查手册.pdf"),
    ("Word 文档", "docs/素材/word/密码重置与账号解锁SOP.docx"),
    ("单图素材", "docs/素材/images/network_topology.png"),
]

print("=" * 80)
print(f"  {'素材':<12}{'text':>6}{'image':>7}{'table':>7}{'分块合计':>10}")
print("=" * 80)
for label, path in cases:
    if not pathlib.Path(path).exists():
        print(f"  [跳过] {path} 不存在")
        continue
    result = asyncio.run(service4.add_document(path, document_id=f"doc-{label}"))
    m = result["modalities"]
    print(f"  {label:<12}{m['text']:>6}{m['image']:>7}{m['table']:>7}{result['chunk_count']:>10}")

print("=" * 80)
print(f"  集合内向量总数：{store4.count()}")

  素材            text  image  table      分块合计
  表格型 PDF          1      0      1         2
  图文混合 PDF         2      2      1         5
  扫描型 PDF          0      2      0         2
  Word 文档          1      1      1         3
  单图素材             0      1      0         1
  集合内向量总数：13


In [12]:
# ============================================================
# Cell 12：看一个图片块的完整元数据（字段对照）
# ============================================================
print("=" * 62)
print("  单图素材 network_topology.png 的分块元数据")
print("=" * 62)
for chunk in store4.list_by_document("doc-单图素材"):
    for key, value in chunk["metadata"].items():
        shown = value if not isinstance(value, str) or len(value) <= 60 else value[:60] + "..."
        print(f"    {key:<14} = {shown!r}")

print()
print("=" * 62)
print("  Word 文档的三类块（按 modality 看字段差异）")
print("=" * 62)
for chunk in store4.list_by_document("doc-Word 文档"):
    meta = chunk["metadata"]
    print(f"    [{meta['modality']:<5}] page={meta.get('page')} "
          f"chunk_index={meta.get('chunk_index')} "
          f"content={(meta.get('content') or '')[:28]!r}")

  单图素材 network_topology.png 的分块元数据
    chunk_index    = 0
    modality       = 'image'
    category       = 'general'
    document_id    = 'doc-单图素材'
    source         = 'network_topology.png'
    context        = 'network_topology.png'
    page           = 1
    asset_id       = 'kb-nb6'
    caption        = ''

  Word 文档的三类块（按 modality 看字段差异）
    [text ] page=1 chunk_index=0 content='密码重置与账号解锁 SOP\n适用场景：企业办公账号忘记密'
    [image] page=1 chunk_index=1 content=''
    [table] page=1 chunk_index=2 content='| 账号类型 | 有效期 | 到期提前提醒 |\n| --'


### 4.4 一个诚实的说明：表格分块

课程大纲里提到"**大表分块时每组重复表头**"。需要说明的是：**当前代码并未实现表格分块**——
`add_document` 把整张 Markdown 表作为**一个 table 块**整体嵌入（见 Part 7 源码）。

- 现状：小表 / 中等表没问题，整表嵌入反而保住了完整行列关系；
- 进阶方向：超大表（几十行以上）可做"按行分组、每组重复表头"的分块，避免单块过长稀释语义。

> 教学原则：**只讲代码里真实存在的东西**。这一条作为课后扩展方向，不当作已实现功能来讲。

---
## Part 5：文本优先嵌入与精排
---

### 5.1 嵌入是什么？为什么"换模型就要重建索引"

**嵌入（Embedding）** 把一段文本映射成一个高维向量，语义相近的文本向量距离也相近。
检索时算"查询向量"与"库内向量"的相似度，取最相似的几条。

关键结论：**不同嵌入模型输出的向量空间不通用**。
本项目向量维度由 `EMBEDDING_DIM` 固定（默认 1024），一旦换模型或改维度，
旧索引与新查询向量对不上，**必须重建全部索引**。

In [13]:
# ============================================================
# Cell 13：与嵌入 / 精排 / 检索相关的配置
# ============================================================
from core.config import settings

print("=" * 62)
print("  嵌入与精排配置（来自 .env / config.py 默认值）")
print("=" * 62)
print(f"  EMBEDDING_MODEL      = {settings.EMBEDDING_MODEL!r}")
print(f"  EMBEDDING_BASE_URL   = {settings.EMBEDDING_BASE_URL!r}")
print(f"  EMBEDDING_DIM        = {settings.EMBEDDING_DIM}")
print(f"  RERANKER_ENABLED     = {settings.RERANKER_ENABLED}")
print(f"  RERANKER_MODEL       = {settings.RERANKER_MODEL!r}")
print(f"  RETRIEVAL_TOP_K      = {settings.RETRIEVAL_TOP_K}   （向量召回的条数）")
print(f"  RERANKER_TOP_N       = {settings.RERANKER_TOP_N}    （精排后保留的条数）")
print(f"  CHROMA_COLLECTION    = {settings.CHROMA_COLLECTION!r}")
print()
print("  提示：这套配置走统一「能力四元组」，换供应商只改 .env，代码零改动。")

  嵌入与精排配置（来自 .env / config.py 默认值）
  EMBEDDING_MODEL      = 'Qwen/Qwen3-VL-Embedding-8B'
  EMBEDDING_BASE_URL   = 'https://api.siliconflow.cn/v1'
  EMBEDDING_DIM        = 1024
  RERANKER_ENABLED     = True
  RERANKER_MODEL       = 'Qwen/Qwen3-VL-Reranker-8B'
  RETRIEVAL_TOP_K      = 10   （向量召回的条数）
  RERANKER_TOP_N       = 3    （精排后保留的条数）
  CHROMA_COLLECTION    = 'mde_knowledge'

  提示：这套配置走统一「能力四元组」，换供应商只改 .env，代码零改动。


### 5.2 EmbeddingClient 的两个关键设计

1. **`dimensions` 截断（Matryoshka）**：`Qwen3-VL-Embedding-8B` 原生 4096 维，
   通过 `dimensions` 参数截断到 `EMBEDDING_DIM=1024`，控制存储与成本；
   若供应商不支持该参数，则降级为原生维度重试一次（并记住这个能力）。
2. **维度自检 `_ensure_dim`**：每次嵌入后校验返回维度 == 配置维度，
   不一致**立即报错**，绝不把不同维度的向量交给 ChromaDB（否则此后每次检索都会失败）。

In [14]:
# ============================================================
# Cell 14：嵌入核心源码
# ============================================================
print("=" * 62); print("  EmbeddingClient._embed_sync —— 批量嵌入 + dimensions 降级"); print("=" * 62)
print(inspect.getsource(embedding.EmbeddingClient._embed_sync))
print("=" * 62); print("  EmbeddingClient._ensure_dim —— 维度自检"); print("=" * 62)
print(inspect.getsource(embedding.EmbeddingClient._ensure_dim))

  EmbeddingClient._embed_sync —— 批量嵌入 + dimensions 降级
    def _embed_sync(self, inputs: list[str]) -> list[list[float]]:
        """同步批量嵌入（文本或 data URL 图片）。"""
        client = self._client_or_raise()
        kwargs: dict = {"model": self._model, "input": inputs, "timeout": _REQUEST_TIMEOUT}
        if self._dimensions and self._use_dimensions:
            kwargs["dimensions"] = self._dimensions

        try:
            resp = client.embeddings.create(**kwargs)
        except Exception as exc:  # 网络 / 鉴权 / 模型等一切异常统一包装
            # 只有「供应商不认 dimensions 参数」才降级为原生维度重试；超时/网络/鉴权等
            # 瞬时故障必须原样抛出——否则会静默退回 4096 维，与按 EMBEDDING_DIM 建好的
            # 索引不兼容，且此后进程内每次检索都失败（实测一次 Request timed out 即触发）
            if "dimensions" not in kwargs or not _is_unsupported_param_error(exc):
                raise EmbeddingError(f"嵌入调用失败：{exc}") from exc
            # 供应商不支持 dimensions 参数：降级为模型原生维度重试一次，并记住该能力
            logger.warning("嵌入接口不支持 dimensions 参数，降级为模型原生维度：%s", exc)
            self._use

### 5.3 一个真实的坑：别把"瞬时故障"当成"参数不支持"

`_is_unsupported_param_error` 的注释里记录了一个踩过的雷，非常值得学：
如果**误判**（比如把一次网络超时当成"供应商不支持 dimensions"），客户端会永久关掉
`dimensions` 参数、退回 4096 维，而索引是按 1024 维建的 —— 结果**之后每次检索都失败**。

所以判断必须"只认异常文本"，且同时命中 "dimension" 与"不支持/未知/非法"语义，
**不能按异常类型放宽**（图片超限、非法输入同样返回 400）。

In [15]:
# ============================================================
# Cell 15：判断函数源码 + 实测三种异常文本
# ============================================================
print(inspect.getsource(embedding._is_unsupported_param_error))

print()
print("实测：")
samples = [
    "Invalid dimensions",                                     # 真的是不支持 dimensions
    "Request timed out",                                      # 瞬时网络故障
    "BadRequestError 400 invalid image",                      # 别的 400 错误
]
for text in samples:
    verdict = embedding._is_unsupported_param_error(Exception(text))
    print(f"  {text!r:<45} -> {verdict}")
print()
print("结论：只有第 1 种才降级为原生维度；后两种会原样抛出，避免静默改变维度。")

def _is_unsupported_param_error(exc: Exception) -> bool:
    """判断异常是否为「供应商不认 dimensions 参数」。

    必须与超时/网络/鉴权等**瞬时**故障区分开：瞬时故障若被误当成「参数不支持」，
    客户端会退回模型原生维度（4096）并与按 EMBEDDING_DIM 建好的索引不匹配，
    且 `_use_dimensions` 被永久置 False，导致进程内所有检索静默失效。

    因此**只认异常文本**，要求同时命中 "dimension" 与「不支持/未知/非法」语义；
    不能按异常类型（如 BadRequestError）放宽——图片超限、非法输入等同样返回 400，
    一旦被误判就会永久关闭 dimensions 参数，使进程内嵌入全线失败。
    """
    text = str(exc).lower()
    return "dimension" in text and any(
        token in text for token in ("unsupported", "not support", "unknown", "invalid")
    )


实测：
  'Invalid dimensions'                          -> True
  'Request timed out'                           -> False
  'BadRequestError 400 invalid image'           -> False

结论：只有第 1 种才降级为原生维度；后两种会原样抛出，避免静默改变维度。


### 5.4 精排：为什么"召回"之后还要"精排"

向量召回（top_k=10）是**粗筛**，快但不精确；精排（Reranker）用交叉编码模型
逐条打分，慢但准，只对少数候选做，取 top_n=3。

- 召回：`encode_text(query)` → ChromaDB 余弦相似度 → top_k；
- 精排：调 `POST {base_url}/rerank`，用 `Qwen3-VL-Reranker-8B` 重排 → top_n。

精排的候选文本怎么取？这里有个关键细节 —— **image 块优先取配图描述 `caption`**，
无描述才回退页正文 `context`（`_candidate_text`）。

In [16]:
# ============================================================
# Cell 16：精排源码 + 候选文本选取
# ============================================================
print("=" * 62); print("  RerankerClient._candidate_text —— 候选文本如何取"); print("=" * 62)
print(inspect.getsource(embedding.RerankerClient._candidate_text))
print("=" * 62); print("  RerankerClient.rerank —— 精排并写回分数"); print("=" * 62)
print(inspect.getsource(embedding.RerankerClient.rerank))

print()
print("实测 _candidate_text：")
with_caption = {"metadata": {"modality": "image", "caption": "配图描述", "context": "页正文"}}
without = {"metadata": {"modality": "image", "context": "页正文"}}
text_hit = {"metadata": {"modality": "text", "content": "正文内容", "context": "x"}}
print("  image 有描述 ->", embedding.RerankerClient._candidate_text(with_caption))
print("  image 无描述 ->", embedding.RerankerClient._candidate_text(without))
print("  text 块      ->", embedding.RerankerClient._candidate_text(text_hit))

  RerankerClient._candidate_text —— 候选文本如何取
    @staticmethod
    def _candidate_text(candidate: dict) -> str:
        """把候选块转成参与精排的可检索文本。

        text / table 取 ``metadata["content"]``；image / page 优先取配图描述 ``caption``、
        无描述时回退 ``context``（页正文）。
        """
        metadata = candidate.get("metadata") or {}
        modality = metadata.get("modality", "text")
        if modality in ("image", "page"):
            return str(
                metadata.get("caption")
                or metadata.get("context")
                or metadata.get("content")
                or ""
            )
        return str(metadata.get("content") or metadata.get("context") or "")

  RerankerClient.rerank —— 精排并写回分数
    async def rerank(
        self, query: str, candidates: list[dict], top_n: int
    ) -> list[dict]:
        """对候选结果精排。

        输入 ``candidates`` 元素形如 ``{"id","score","metadata"}``，返回同结构列表，
        并把精排分数写回 ``score``，按分数降序取前 ``top_n``。
        接口不可用时记录日志并回退到原顺序（不抛异常）。
        """
   

### 5.5 精排失败要"软回退"，不能拖垮主链路

精排是一次**外部网络调用**，可能超时/失败。若让它抛异常，整个检索就挂了。
看源码你会发现 `rerank()` 在异常和空返回时都 `logger.warning` 后**回退到向量召回顺序**，
这就是"可靠性优先"的工程取向。

In [17]:
# ============================================================
# Cell 17：维度自检实测（离线）
# ============================================================
c = embedding.EmbeddingClient(base_url="http://demo", api_key="k", model="m", dimensions=8)
# 假装模型只返回 4 维（与配置的 8 维不一致）
c._embed_sync = lambda inputs: [[0.1] * 4 for _ in inputs]
try:
    asyncio.run(c.encode_text("维度自检"))
    print("  [异常] 维度不一致竟然没被拦截")
except embedding.EmbeddingError as e:
    print(f"  [OK] 维度不一致被立即拦截：{e}")

print()
print(f"  精排开关 RERANKER_ENABLED = {settings.RERANKER_ENABLED}")
print("  关闭时 rerank() 直接返回 candidates[:top_n]，不发起网络请求；")
print("  开启但调用失败时，同样回退向量召回顺序（软回退，不抛异常）。")

  [OK] 维度不一致被立即拦截：嵌入维度不一致：配置 EMBEDDING_DIM=8，模型实际返回 4。请对齐 EMBEDDING_DIM 后重建索引（见设计文档 7.2 / 17.3）。

  精排开关 RERANKER_ENABLED = True
  关闭时 rerank() 直接返回 candidates[:top_n]，不发起网络请求；
  开启但调用失败时，同样回退向量召回顺序（软回退，不抛异常）。


---
## Part 6：单集合向量库（ChromaDB）
---

### 6.1 为什么是"一个集合"

文本优先方案下，正文 / 表格 / 图片描述都在**同一个文本向量空间**，
所以只需要**一个集合、一次查询**就能覆盖全部模态，跨模态相似度可直接比较排序。

- 选型：ChromaDB **嵌入式**（零容器、零配置），持久化在 `data/chroma`；
- 代价：嵌入式模式是**单进程独占**，服务必须单实例运行，不能开多 worker；
- 距离：集合用 `hnsw:space = cosine`，检索时把"余弦距离"换算成分数 `score = 1 - distance`。

In [18]:
# ============================================================
# Cell 18：VectorStore 的核心方法
# ============================================================
print((vstore.__doc__ or "").strip())
print()
print("=" * 62); print("  add_batch —— 批量写入"); print("=" * 62)
print(inspect.getsource(vstore.VectorStore.add_batch))
print("=" * 62); print("  search —— 向量检索（score = 1 - cosine 距离）"); print("=" * 62)
print(inspect.getsource(vstore.VectorStore.search))

ChromaDB 向量库封装（单集合、单维度），见设计文档 7.2 / 7.3 / 7.6。

采用**文本优先（完全文本化）**方案：文本 / 表格 / 图片描述都以文本嵌入映射到同一
向量空间，因此只需一个集合、一次查询即可覆盖全部模态，跨模态相似度可直接排序比较。
图片原图不参与检索，存 `data/media` 并以 ``asset_id`` 关联，检索命中后作附件回传。

对外单例：``vector_store``。

  add_batch —— 批量写入
    def add_batch(
        self,
        embeddings: list[list[float]],
        metadatas: list[dict],
        chunk_ids: Optional[list[str]] = None,
    ) -> list[str]:
        """批量写入向量，返回 chunk_id 列表。"""
        if not embeddings:
            return []
        if len(embeddings) != len(metadatas):
            raise ValueError("embeddings 与 metadatas 数量不一致。")
        if chunk_ids is None:
            chunk_ids = [f"chunk-{uuid.uuid4().hex[:16]}" for _ in embeddings]
        if len(chunk_ids) != len(embeddings):
            raise ValueError("chunk_ids 与 embeddings 数量不一致。")

        collection = self._get_collection()
        collection.add(
            ids=chunk_ids,
            embeddings=embeddings,
            metadatas=[_sanitize_metadata(m) for m 

### 6.2 元数据必须"可序列化"：`_sanitize_metadata`

ChromaDB 的元数据只接受 `str / int / float / bool`。本项目入库的元数据里可能有
`None`、`list`、`dict`，所以用一个清洗函数统一规整：`None → ""`，`list/dict → JSON 字符串`。

In [19]:
# ============================================================
# Cell 19：元数据清洗 + 实测
# ============================================================
print(inspect.getsource(vstore._sanitize_metadata))

raw = {"modality": "image", "page": 3, "tags": ["vpn", "网络"], "extra": None, "ok": True}
print()
print("  原始:", raw)
print("  清洗:", vstore._sanitize_metadata(raw))
print("  → None 变空串、list 变 JSON 字符串，其余标量原样保留。")

def _sanitize_metadata(metadata: dict) -> dict:
    """把 metadata 规整为「可序列化标量」字典。

    规则：None -> ""；dict/list -> JSON 字符串；其余标量原样保留。
    ChromaDB 仅接受 str/int/float/bool 类型的元数据值。
    """
    clean: dict[str, Any] = {}
    for key, value in (metadata or {}).items():
        if value is None:
            clean[str(key)] = ""
        elif isinstance(value, (str, int, float, bool)):
            clean[str(key)] = value
        elif isinstance(value, (dict, list, tuple)):
            clean[str(key)] = json.dumps(value, ensure_ascii=False)
        else:
            clean[str(key)] = str(value)
    return clean


  原始: {'modality': 'image', 'page': 3, 'tags': ['vpn', '网络'], 'extra': None, 'ok': True}
  清洗: {'modality': 'image', 'page': 3, 'tags': '["vpn", "网络"]', 'extra': '', 'ok': True}
  → None 变空串、list 变 JSON 字符串，其余标量原样保留。


In [20]:
# ============================================================
# Cell 20：VectorStore 增 / 查 / 删 实测
# ============================================================
import tempfile
import pathlib

tmp = tempfile.mkdtemp()
st = vstore.VectorStore(path=str(pathlib.Path(tmp) / "chroma"), collection_name="nb36_vs")

st.add_batch(
    embeddings=[[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0]],
    metadatas=[
        {"modality": "text", "document_id": "d1", "content": "VPN 排查"},
        {"modality": "image", "document_id": "d1", "content": "拓扑图"},
        {"modality": "text", "document_id": "d2", "content": "密码重置"},
    ],
    chunk_ids=["c1", "c2", "c3"],
)
print(f"  写入后 count = {st.count()}")

print("\n  检索 [1,0,0,0]（最接近 c1）：")
for hit in st.search([1.0, 0.0, 0.0, 0.0], top_k=2):
    print(f"    {hit['id']}  score={hit['score']:.4f}  modality={hit['metadata']['modality']}")

print("\n  按 document_id=d1 列出分块：")
for hit in st.list_by_document("d1"):
    print(f"    {hit['id']}  {hit['metadata']['content']}")

removed = st.delete_by_document("d1")
print(f"\n  删除 d1：清理 {removed} 条，剩余 count = {st.count()}")
print("  → score = 1 - 余弦距离；where 过滤（如 document_id / modality）在集合内直接完成。")

  写入后 count = 3

  检索 [1,0,0,0]（最接近 c1）：
    c1  score=1.0000  modality=text
    c3  score=0.0000  modality=text

  按 document_id=d1 列出分块：
    c1  VPN 排查
    c2  拓扑图

  删除 d1：清理 2 条，剩余 count = 1
  → score = 1 - 余弦距离；where 过滤（如 document_id / modality）在集合内直接完成。


---
## Part 7：检索编排与结果装配（本节核心）
---

### 7.1 入库：`add_document` 的三步

`add_document` 把 Part 3、4、5 的能力串起来，对**每一页**依次处理三类块：

```
for page in pages:
    ① 文本块：_chunk_text(正文)          → encode_text → 写库
    ② 图片块：downsample → save_media    → 生成 caption → encode_text(caption/context/源名) → 写库
    ③ 表格块：Markdown                   → encode_text → 写库
最后：db.upsert_document(...) 记录文档元数据
```

In [21]:
# ============================================================
# Cell 21：入库主流程源码 add_document
# ============================================================
print(inspect.getsource(mrag.MultimodalRAGService.add_document))

    async def add_document(
        self,
        file_path: str,
        category: str = "general",
        document_id: Optional[str] = None,
    ) -> dict:
        """文档入库：解析 → 分块（文本/图片/表格）→ 统一嵌入 → 落库（向量 + 元数据）。

        返回 ``{"document_id","modalities":{"text":n,"image":n,"table":n},"chunk_count":n}``。
        """
        pages = await parse_document(file_path)
        source_name = Path(file_path).name
        doc_id = document_id or f"doc-{uuid.uuid4().hex[:8]}"

        counts = {"text": 0, "image": 0, "table": 0}
        chunk_index = 0

        for page in pages:
            # 1. 文本块
            for chunk in self._chunk_text(page.text):
                vector = await self.embedder.encode_text(chunk)
                metadata = {
                    "modality": "text",
                    "content": chunk,
                    "source": source_name,
                    "page": page.no,
                    "document_id": doc_id,
                    "category": category,
          

### 7.2 检索：`search` 的两种入口

- **文本查询** `query=`：直接嵌入召回（图片块靠它的**配图描述**参与）——这是"**文查图**"主路径；
- **图片查询** `query_image=`：先由 VLM 生成一段描述，再用描述做文本召回 ——"**图查文 / 图查图**"路径；
  VLM 未配置或失败时**返回空结果**（宁可查不到，也不误召回）。

两条路径最后都汇合到：`召回 top_k → 精排 top_n`。
`text_only=True` 时只召回 text 块（用于对比"纯文本 RAG"的效果差异）。

In [22]:
# ============================================================
# Cell 22：检索编排源码 search + 过滤条件 _build_where
# ============================================================
print(inspect.getsource(mrag.MultimodalRAGService.search))
print("=" * 62); print("  _build_where —— 构造 ChromaDB 过滤条件"); print("=" * 62)
print(inspect.getsource(mrag.MultimodalRAGService._build_where))

    async def search(
        self,
        query: Optional[str] = None,
        query_image: Optional[bytes] = None,
        top_k: Optional[int] = None,
        text_only: bool = False,
        category: Optional[str] = None,
    ) -> list[dict]:
        """多模态检索（单集合、文本优先）：召回 → 精排。

        - 纯文本 ``query``：直接嵌入召回（image 块以其**配图描述**参与），精排后取
          ``RERANKER_TOP_N`` 条 —— 这是「文查图」的主路径；
        - ``query_image``：先由 VLM 生成一段描述，再用该描述做一次文本召回并精排 —— 这是
          「图查文 / 图查图」的路径；VLM 未配置或调用失败时图片查询返回空结果；
        - ``text_only=True`` 时限定只召回 text 块（image / table 块均不参与，
          纯文本模式对比；见 7.6）。
        """
        if not query and not query_image:
            raise ValueError("检索失败：query 与 query_image 至少需要提供一个。")

        limit = top_k or settings.RETRIEVAL_TOP_K
        where = self._build_where(text_only=text_only, category=category)

        if query_image:
            query = await describe_knowledge_image(query_image)
            if not query:
                logger.warning("图片查询无可用描述（VLM 未配

### 7.3 结果装配：三种输出，服务三种用途

检索回来的原始结果（id/score/metadata）不能直接丢给 LLM/前端，要"装配"：

| 方法 | 产出 | 用途 |
|------|------|------|
| `format_results` | 带来源标注的**上下文文本** | 拼进 LLM 提示词 |
| `build_attachments` | 命中**图片的附件列表**（`/api/media/{id}`） | 前端渲染缩略图 |
| `build_citations` | **参考来源**列表（文件/页码/模态） | 前端"参考来源"展示 |

装配时，image 块的正文优先取 `caption`（描述），回退 `context`；text/table 取 `content`。

In [23]:
# ============================================================
# Cell 23：结果装配源码
# ============================================================
print("=" * 62); print("  format_results"); print("=" * 62)
print(inspect.getsource(mrag.MultimodalRAGService.format_results))
print("=" * 62); print("  build_attachments"); print("=" * 62)
print(inspect.getsource(mrag.MultimodalRAGService.build_attachments))
print("=" * 62); print("  build_citations"); print("=" * 62)
print(inspect.getsource(mrag.MultimodalRAGService.build_citations))

  format_results
    def format_results(self, results: list[dict]) -> str:
        """把检索结果拼成带来源标注的上下文。

        text / table 取 ``content``；image / page 优先取配图描述 ``caption``、回退 ``context``；
        每条前缀 ``[序号] 来源：文件（第 N 页·模态）``。
        """
        blocks: list[str] = []
        for index, result in enumerate(results, 1):
            metadata = result.get("metadata") or {}
            body = (
                metadata.get("content")
                or metadata.get("caption")
                or metadata.get("context", "")
            )
            blocks.append(
                f"[{index}] 来源：{metadata.get('source', '未知')}"
                f"（第{metadata.get('page', '-')}页·{metadata.get('modality', '?')}）\n{body}"
            )
        return "\n\n".join(blocks)

  build_attachments
    def build_attachments(self, results: list[dict]) -> list[dict]:
        """把检索命中的图像块转为消息附件（见 8.4）。

        仅 ``modality == "image"`` 的输出，``url`` 走鉴权媒体接口，并附带配图描述 ``caption``。
        """
        attachment

### 7.4 端到端实测：一条"文搜图"闭环

下面用假嵌入器 + 假描述（模拟 VLM 返回）跑通：
**图片入库（配图描述进文本空间）→ 用文字查询命中图片块 → 装配上下文/附件/来源**。

关键点：入库与查询用**同一段描述文本**，所以假嵌入器产出的向量完全一致，命中是确定的。

In [24]:
# ============================================================
# Cell 24：端到端 —— 文搜图（离线，复现 tests/test_rag.py 核心场景）
# ============================================================
import tempfile
import pathlib
import asyncio

from core.rag.multimodal_rag import MultimodalRAGService

CAPTION = "企业远程办公网络拓扑图，含 VPN 网关与分支节点"

# 复用 Part 4 的 stub（落盘 / 元数据），并让"配图描述"返回固定文案
mrag.db = FakeDB()
mrag.save_media = _stub_save

async def _caption(image_bytes):
    return CAPTION

mrag.describe_knowledge_image = _caption

tmp = tempfile.mkdtemp()
store7 = vstore.VectorStore(path=str(pathlib.Path(tmp) / "chroma"), collection_name="nb36_part7")
svc = MultimodalRAGService(embedder=FakeEmbedder(), reranker=DisabledReranker(), vector_store=store7)

asyncio.run(svc.add_document("docs/素材/images/network_topology.png", document_id="doc-topo"))
print(f"  入库完成，集合内向量数：{store7.count()}")

print("\n【文搜图】用文字查询命中知识库配图块：")
hits = asyncio.run(svc.search(query=CAPTION))
for hit in hits:
    meta = hit["metadata"]
    print(f"  modality={meta['modality']} score={hit['score']:.3f} source={meta['source']} "

          f"asset_id={meta.get('asset_id')}")

print("\n  ① 装配成上下文（喂给 LLM）：")
print(svc.format_results(hits))

print("\n  ② 附件（原图回传前端）：")
for att in svc.build_attachments(hits):
    print("   ", att)

print("\n  ③ 参考来源（前端展示）：")
for cite in svc.build_citations(hits):
    print("   ", cite)

  入库完成，集合内向量数：1

【文搜图】用文字查询命中知识库配图块：
  modality=image score=1.000 source=network_topology.png asset_id=kb-nb7

  ① 装配成上下文（喂给 LLM）：
[1] 来源：network_topology.png（第1页·image）
企业远程办公网络拓扑图，含 VPN 网关与分支节点

  ② 附件（原图回传前端）：
    {'type': 'image', 'url': '/api/media/kb-nb7', 'source': 'network_topology.png', 'page': 1, 'caption': '企业远程办公网络拓扑图，含 VPN 网关与分支节点'}

  ③ 参考来源（前端展示）：
    {'source': 'network_topology.png', 'page': 1, 'modality': 'image'}


In [25]:
# ============================================================
# Cell 25：图片查询在"VLM 未配置"时的行为（返回空，不误召回）
# ============================================================
async def _empty(image_bytes):
    return ""

mrag.describe_knowledge_image = _empty

results = asyncio.run(svc.search(query_image=b"topology-bytes"))
print(f"  VLM 未配置（描述为空）时，图片查询返回：{results}")
print("  → 图片查询宁可为空，也不拿无关文本凑数，避免答非所问。")

# 恢复被替换的模块级对象，避免影响后续 Cell
mrag.db = _orig_db
mrag.save_media = _orig_save
mrag.describe_knowledge_image = _orig_caption
print("\n  已恢复模块原始状态。")

图片查询无可用描述（VLM 未配置或调用失败），返回空结果。


  VLM 未配置（描述为空）时，图片查询返回：[]
  → 图片查询宁可为空，也不拿无关文本凑数，避免答非所问。

  已恢复模块原始状态。


### 7.5 索引管理：什么时候必须重建

`rebuild_index()` 会**清空集合**，再重跑 `data/knowledge` 下所有文件。
以下情况**必须**重建，否则新旧向量不兼容、检索会出错或质量骤降：

- **换嵌入模型** / 改 `EMBEDDING_DIM`（向量空间变了）；
- 改了索引语义（比如从"图片直接嵌入"切到"文本优先"）。

注意源码里还处理了一个细节：重建前先读出旧分类、并清空文档元数据表，
否则重建会用新 `document_id` 重新落库，导致知识库列表出现**重复行**。

In [26]:
# ============================================================
# Cell 26：重建索引源码 rebuild_index
# ============================================================
print(inspect.getsource(mrag.MultimodalRAGService.rebuild_index))

    async def rebuild_index(self, force: bool = False) -> dict:
        """全量重建索引：清空集合后重跑 ``data/knowledge`` 下所有文件。

        注意：换嵌入模型、改维度或改索引语义后必须执行（见 7.6）。
        返回 ``{"documents": n, "chunks": n}``。
        """
        self.vector_store.reset()

        # 读出旧元数据用于继承分类，并清空文档表——否则重建会用新 id 重新落库，
        # 导致知识库文档列表出现重复行（向量已清空，旧行不再对应任何分块）。
        category_by_name: dict[str, str] = {}
        try:
            for doc in await db.list_documents():
                filename = doc.get("filename") or ""
                if filename:
                    category_by_name.setdefault(
                        Path(filename).name, doc.get("category") or "general"
                    )
                await db.delete_document(doc["id"])
        except Exception as exc:
            logger.warning("重置文档元数据失败，分类将默认 general：%s", exc)

        files: list[Path] = []
        if KNOWLEDGE_DIR.exists():
            for path in sorted(KNOWLEDGE_DIR.rglob("*")):
                if path.is_file() and path.suf

---
## Part 8：与 Agent 融合 —— 检索即工具
---

### 8.1 为什么检索要做成"工具"，而不是写死在流程里

Agent 需要**自主决定**"这一步要不要查知识库、用什么词查"。所以检索被封装成一个
CrewAI `Tool`（`MultimodalSearchTool`），由 Agent 按需调用：

- 属于**中台内部能力**，以进程内 Tool 接入，**不额外起 MCP Server**（工单才是外部系统，走 MCP）；
- 每次调用都写入 `tool_calls`，满足"全程可旁观"的可观测要求；
- `recorder` 旁路记录命中结果，供会话路由装配 `citations / attachments`。

> ⏳ 提示：下一个 Cell 首次 `import core.ai.crew_tools` 会连带加载 CrewAI / litellm，
> **首次导入可能明显偏慢**（离线环境下 litellm 会尝试联网拉取模型价格表而等待超时），
> 请耐心等待；导入成功后，后续 Cell 会很快。

In [27]:
# ============================================================
# Cell 27：检索工具源码 MultimodalSearchTool
# ============================================================
from core.ai.crew_tools import MultimodalSearchTool
print(inspect.getsource(MultimodalSearchTool))

class MultimodalSearchTool(BaseTool):
    """多模态知识库检索：文本与图片共用统一向量空间。"""

    name: str = "multimodal_search"
    description: str = (
        "在IT知识库中检索问题解决方案，支持文字与图片两种查询方式。"
        "用户用文字描述问题时填写 query（文查文/文查图）；"
        "用户提供报错截图时填写 image_path（图查文，找相关文本与相似截图）。"
        "返回带来源标注（文件+页码+模态）的知识片段。"
    )
    args_schema: Type[BaseModel] = MultimodalSearchArgs
    recorder: Any = Field(default=None, exclude=True)

    def _run(self, query: Optional[str] = None, image_path: Optional[str] = None) -> str:
        started = time.perf_counter()
        image_bytes = _load_image(image_path) if image_path else None
        if not query and not image_bytes:
            return "检索失败：请至少提供 query 或 image_path 之一。"
        try:
            results = run_sync(multimodal_rag.search(query=query, query_image=image_bytes))
        except Exception as exc:  # noqa: BLE001
            _log_tool_call(self.recorder, self.name, {"query": query, "image_path": image_path},
                           str(exc), st

### 8.2 一个真实 Bug 的修复：只在新结果"更优"时覆盖

`MultimodalSearchTool._run` 里有一段很关键的判断：

```python
if not self.recorder.retrieved or new_top > self.recorder.top_score:
    self.recorder.retrieved = results
    self.recorder.top_score = new_top
```

**为什么不能无条件覆盖？** 因为 Agent 可能**自行编造查询词**发起一次低分检索。
例如：用户上传了拓扑图（图片自匹配高分 0.88），Agent 却凭空去搜"错误码619"，返回 0.06。
若无条件覆盖，就会把高分命中顶掉，最终答案与参考来源"答非所问"。

In [28]:
# ============================================================
# Cell 28：复现"保留最优命中"（离线）
# ============================================================
import core.ai.crew_tools as crew_tools
from core.service.turn_context import TurnRecorder

_orig_tool_rag = crew_tools.multimodal_rag


class StubRAG:
    """替掉真实 multimodal_rag，固定返回给定结果，专测覆盖逻辑。"""

    def __init__(self, results):
        self.results = results

    async def search(self, query=None, query_image=None, **kwargs):
        return self.results

    def format_results(self, results):
        return "stub"


recorder = TurnRecorder(conversation_id="c-1")
tool = MultimodalSearchTool(recorder=recorder)

high = [{"id": "img", "score": 0.88, "metadata": {"modality": "image"}}]
low = [{"id": "txt", "score": 0.06, "metadata": {"modality": "text"}}]

crew_tools.multimodal_rag = StubRAG(high)
tool._run(query="看这张拓扑图")
print(f"  第一次（图片自匹配，高分 0.88）：top_score={recorder.top_score}，命中={recorder.retrieved[0]['id']}")

crew_tools.multimodal_rag = StubRAG(low)
tool._run(query="VPN连接错误码619处理建议")
print(f"  第二次（Agent 凭空低分 0.06）：top_score={recorder.top_score}，命中={recorder.retrieved[0]['id']}")

print("  → 低分没有覆盖高分：保留最优命中，答案才不会被带偏。")

crew_tools.multimodal_rag = _orig_tool_rag
print("\n  已恢复 crew_tools.multimodal_rag。")

  第一次（图片自匹配，高分 0.88）：top_score=0.88，命中=img
  第二次（Agent 凭空低分 0.06）：top_score=0.88，命中=img
  → 低分没有覆盖高分：保留最优命中，答案才不会被带偏。

  已恢复 crew_tools.multimodal_rag。


---
## Part 9：实战任务 + 随堂练习（含答案）
---

### 9.1 实战任务：管理端知识库接口

本节所有能力，最终通过一组 admin 接口对外提供服务（`core/api/admin_api.py`）：

| 接口 | 作用 |
|------|------|
| `POST /api/admin/knowledge/upload` | 上传文档 → 解析入库 |
| `GET  /api/admin/knowledge` | 列出已入库文档 |
| `GET  /api/admin/knowledge/{id}/chunks` | 查看某文档的全部分块（含模态/页码） |
| `POST /api/admin/knowledge/rebuild` | 全量重建索引 |
| `POST /api/admin/knowledge/test_search` | 检索测试（多模态 vs 纯文本对比） |
| `POST /api/admin/knowledge/seed` | 一键加载演示知识库 |
| `GET  /api/admin/knowledge/stats` | 向量库统计 |

In [29]:
# ============================================================
# Cell 29：检索测试接口源码 test_search（含多模态 / 纯文本切换）
# ============================================================
from core.api import admin_api

print(inspect.getsource(admin_api.TestSearchBody))
print(inspect.getsource(admin_api.test_search))
print("要点：")
print("  - mode='text_only' 透传到 service.search(text_only=True)，用于对比纯文本 RAG 的召回差异")
print("  - 图片分块返回 asset_id 与可访问 url，前端可直接渲染")

class TestSearchBody(BaseModel):
    """POST /knowledge/test_search 的请求体。

    - query：检索文本（可为空，仅给图片时也能检索）；
    - mode：multimodal（多模态）或 text_only（仅文本）；
    - image_url：以图搜图的截图 URL（可选）。
    """

    query: str = ""
    mode: str = "multimodal"  # multimodal | text_only
    image_url: str = ""

@router.post("/knowledge/test_search")
async def test_search(body: TestSearchBody, user: dict = Depends(require_role("admin"))) -> list[dict]:
    """检索测试：多模态 vs 纯文本对比（见 7.6 核心演示点）。

    调用方：管理端知识库「检索测试」面板。
    入参：请求体 TestSearchBody。
    鉴权：仅 admin。
    校验：query 与截图至少提供一个，否则 400。
    错误码：400 —— 两者皆空；500 —— 检索异常。
    返回：命中列表；score 保留 4 位小数，content 截断到 300 字，
          图片分块附带 asset_id 与可访问 url。
    """
    query_image: Optional[bytes] = None
    if body.image_url:
        asset_id = body.image_url.rstrip("/").split("/")[-1]
        path = media_file(asset_id)
        if path is not None:
            query_image = path.read_bytes()
    if not body.query and query_image is None:
        raise HTTPExce

### 9.2 常见误区与答疑

| 误区 | 纠正 |
|------|------|
| "多模态 = 多集合 / 多向量空间" | 本项目用**单一文本空间**，靠"图片转描述"把多模态统一到文本 |
| "命中图片只回正文就行" | 图片块正文优先取 `caption`；命中图要把**原图作为附件**回传 |
| "换了嵌入模型直接重启就行" | 向量空间变了，**必须重建索引**，否则检索全错 |
| "精排失败就让检索报错" | 精排是增强，失败要**软回退**到向量召回顺序，主链路不能断 |
| "低分检索也能覆盖之前的高分命中" | 只在**新结果更优**时覆盖，防止 Agent 编词把好命中冲掉 |

### 9.3 随堂练习一：文本优先下，如何近似实现"以图搜图"

产品说："用户上传一张图，能不能找到知识库里**相似的图**？"
在"文本优先"方案（图片不直接嵌入）下，请给出你的实现思路，并说明代价。

In [30]:
# ============================================================
# Cell 30：练习一参考思路（含答案）
# ============================================================
answer = """
思路：把"图搜图"近似成"描述搜图"。

1. 入库期：每张图已生成并存入 caption（配图描述），描述向量在文本集合里；
2. 查询期：对用户图调用 describe_knowledge_image() 得到描述 →
   encode_text(描述) → 在向量库里召回；
3. 过滤：召回后只保留 modality == 'image' 的结果，即"相似的图"；
4. 返回：命中的原图通过 build_attachments() 作为附件回传。

代价：
  - 相似度取决于"描述是否抓住了图的关键语义"，不是像素级相似；
  - VLM 未配置时该功能不可用（描述为空 → 返回空结果，不误召回）。

对应代码：multimodal_rag.search(query_image=...) 已经实现了 1~3 步，
build_attachments() 负责第 4 步。
"""
print(answer)


思路：把"图搜图"近似成"描述搜图"。

1. 入库期：每张图已生成并存入 caption（配图描述），描述向量在文本集合里；
2. 查询期：对用户图调用 describe_knowledge_image() 得到描述 →
   encode_text(描述) → 在向量库里召回；
3. 过滤：召回后只保留 modality == 'image' 的结果，即"相似的图"；
4. 返回：命中的原图通过 build_attachments() 作为附件回传。

代价：
  - 相似度取决于"描述是否抓住了图的关键语义"，不是像素级相似；
  - VLM 未配置时该功能不可用（描述为空 → 返回空结果，不误召回）。

对应代码：multimodal_rag.search(query_image=...) 已经实现了 1~3 步，
build_attachments() 负责第 4 步。



### 9.4 随堂练习二：`text_only` 过滤 与"保留最优"

请回答两个问题：
1. `search(text_only=True)` 内部是如何把"只召回 text 块"翻译成 ChromaDB 查询条件的？
2. 为什么 `MultimodalSearchTool` 要"只在新结果更高分时才覆盖"？

> 提示：第 1 问去看 `_build_where`；第 2 问想想 Agent 自己编查询词的场景。

In [31]:
# ============================================================
# Cell 31：练习二参考思路（含答案）
# ============================================================
from core.rag.multimodal_rag import MultimodalRAGService

svc = MultimodalRAGService()
print("1) _build_where 把 text_only / category 翻译成 ChromaDB 过滤条件：")
print("   text_only=True, 无分类   ->", svc._build_where(text_only=True, category=None))
print("   text_only=False, 分类=network ->", svc._build_where(text_only=False, category="network"))
print("   text_only=True, 分类=network  ->", svc._build_where(text_only=True, category="network"))
print()
print("   说明：多个条件用 {'$and': [...]} 组合；modality 用 {'$in': ['text']} 表达。")
print()
print("2) 为什么要'只在新结果更优时覆盖'：")
print("   - Agent 可能自行编造查询词，发起一次低分检索；")
print("   - 若无条件覆盖，高分命中（如图片自匹配 0.88）会被低分（0.06）顶掉；")
print("   - 结果：最终回答与参考来源'答非所问'。")
print("   → 因此只在 new_top > recorder.top_score 时才更新命中（保留最优）。")

1) _build_where 把 text_only / category 翻译成 ChromaDB 过滤条件：
   text_only=True, 无分类   -> {'modality': {'$in': ['text']}}
   text_only=False, 分类=network -> {'category': 'network'}
   text_only=True, 分类=network  -> {'$and': [{'modality': {'$in': ['text']}}, {'category': 'network'}]}

   说明：多个条件用 {'$and': [...]} 组合；modality 用 {'$in': ['text']} 表达。

2) 为什么要'只在新结果更优时覆盖'：
   - Agent 可能自行编造查询词，发起一次低分检索；
   - 若无条件覆盖，高分命中（如图片自匹配 0.88）会被低分（0.06）顶掉；
   - 结果：最终回答与参考来源'答非所问'。
   → 因此只在 new_top > recorder.top_score 时才更新命中（保留最优）。


### 9.5 本节小结

| 环节 | 文件 | 核心设计 |
|------|------|----------|
| 解析 | `parsers.py` | 统一 `ParsedPage`；注册表路由；扫描件整页保留；表格转 Markdown |
| 分块 | `multimodal_rag.py` | 文本 500/50 滑窗；图片三级兜底载体（描述→正文→文件名） |
| 嵌入 | `embedding.py` | 文本优先单空间；`dimensions` 截断 + 维度自检；错误判断只认异常文本 |
| 精排 | `embedding.py` | 召回 top_k → 精排 top_n；候选 image 优先取 caption；失败软回退 |
| 存储 | `vector_store.py` | ChromaDB 单集合 cosine；`score = 1 - 距离`；元数据清洗为标量 |
| 检索 | `multimodal_rag.py` | 文本/图片两入口；`text_only` 过滤；结果装配（上下文/附件/来源） |
| Agent | `crew_tools.py` | 检索即工具；调用可观测；只在新结果更优时覆盖 |

### 9.6 交付物 / 验收

- 多格式文档（PDF/Word/PNG/MD）可解析、分块、入库；
- 管理端 `POST /api/admin/knowledge/{upload,rebuild,test_search,seed}` 可用；
- **文搜图**：文本 query 能命中知识库配图块的 caption；
- **图片查询**：截图 → 描述 → 文本匹配 → 带图回复闭环；
- `tests/test_rag.py` 全绿（解析 / 分块 / 降采样 / 增删查 / 结果装配 / 保留最优命中）。

### 9.7 下节课预告

**第 37 节：多模态与多智能体协同开发实现**
- CrewAI 三角色编排（一线客服 / 工单专家 / 质检专员）；
- 一次 LLM 调用完成意图 + 情绪识别；
- 转人工判定四条规则 + 人机协同闭环 + MCP 工单系统。

---

## 📚 参考资料

- 课件：`第36节-多模态与RAG智能体深度融合开发.pptx`
- 代码：`core/rag/parsers.py` —— 统一解析管线
- 代码：`core/rag/embedding.py` —— 文本优先嵌入与精排
- 代码：`core/rag/vector_store.py` —— ChromaDB 单集合
- 代码：`core/rag/multimodal_rag.py` —— 入库 / 检索 / 结果装配
- 代码：`core/ai/crew_tools.py` —— 检索即工具
- 代码：`core/api/admin_api.py` —— 知识库管理接口
- 测试：`tests/test_rag.py`

> 🎉 恭喜你完成了本节课的学习！
> 现在你不仅知道代码怎么写，更知道**为什么选"文本优先"、为什么这样做取舍**。